In [2]:
import numpy as np
import pandas as pd

In [30]:
biomarkers = ['KLRD1','FCGR1A','NKG7','CXCL11','CXCL9','PRF1','MSR1',
'EOMES','APOL3','SAMD3','IRF8','GBP1','HLA-F','RUNX3','STAT1','TRAF5','CSF2RA',
'CCL5','HCP5','ENTPD1']

cell_cycle = ['ANLN','ASPM','BUB1','BUB1B','CCNA2','CCNB1','CDKN3','CENPA','CENPF','CEP55','DLGAP5','DTL','EZH2','KIAA0101','KIF11','KIF20A','MKI67','NUSAP1','RRM2','TOP2A','TPX2','TTK','TYMS','UBE2C']

In [19]:
degs_liver = pd.read_excel("./data/degs_result_liver.xlsx")
degs_liver.index = degs_liver['Gene']
degs_lung = pd.read_excel("./data/degs_result_lung.xlsx")
degs_lung.index = degs_lung['Gene']
degs_heart = pd.read_excel("./data/degs_result_heart.xlsx")
degs_heart.index = degs_heart['Gene']
degs_kidney = pd.read_excel("./data/degs_result_kidney.xlsx")
degs_kidney.index  = degs_kidney['Gene']

In [31]:
# List of all your result dataframes
deg_tables = {
    'liver': degs_liver,
    'lung': degs_lung,
    'heart': degs_heart,
    'kidney': degs_kidney
}

In [32]:
biomarker_subsets = {}
cell_cycle_subsets = {}

for organ, df in deg_tables.items():    
    existing_biomarkers = df.index.intersection(biomarkers)
    biomarker_subsets[organ] = df.loc[existing_biomarkers]
    
    existing_cc = df.index.intersection(cell_cycle)
    cell_cycle_subsets[organ] = df.loc[existing_cc]
    
    print(f"{organ.capitalize()}: Found {len(existing_biomarkers)} biomarkers and {len(existing_cc)} cell cycle genes.")


Liver: Found 20 biomarkers and 24 cell cycle genes.
Lung: Found 20 biomarkers and 24 cell cycle genes.
Heart: Found 20 biomarkers and 24 cell cycle genes.
Kidney: Found 20 biomarkers and 24 cell cycle genes.


In [33]:
def get_status(logFC, adj_P):
    if adj_P >= 0.05:
        return "non-significant"
    elif logFC > 0:
        return "upregulated"
    else:
        return "downregulated"

# Iterate through each organ and create a new status column
organs = ['liver', 'kidney', 'heart', 'lung']

for organ in organs:
    logFC_col = f'{organ}.logFC'
    adjP_col = f'{organ}.adj.P.Val'
    status_col = f'{organ}.status'
    
    # Apply the status function row by row
    combined_biomarkers[status_col] = combined_biomarkers.apply(
        lambda x: get_status(x[logFC_col], x[adjP_col]), axis=1
    )

# Save the updated table with status columns
combined_biomarkers.to_csv("./data/biomarker_status_results.csv", index=False)

# Display a preview focusing on the new status columns
print(combined_biomarkers[['Gene', 'liver.status', 'kidney.status', 'heart.status', 'lung.status']].head())

     Gene liver.status kidney.status heart.status  lung.status
0  FCGR1A  upregulated   upregulated  upregulated  upregulated
1    MSR1  upregulated   upregulated  upregulated  upregulated
2    GBP1  upregulated   upregulated  upregulated  upregulated
3   STAT1  upregulated   upregulated  upregulated  upregulated
4  CXCL11  upregulated   upregulated  upregulated  upregulated


In [34]:
new_column_order = [
    'Gene',
    'liver.logFC', 'liver.adj.P.Val', 'liver.status',
    'kidney.logFC', 'kidney.adj.P.Val', 'kidney.status',
    'heart.logFC', 'heart.adj.P.Val', 'heart.status',
    'lung.logFC', 'lung.adj.P.Val', 'lung.status'
]

combined_biomarkers = combined_biomarkers[new_column_order]

combined_biomarkers.to_excel("./data/biomarker_final_report.xlsx", index=False)